# Publishing a Notebook to the tool registry
This notebook showcases how to use the tool-registry REST API to publish a tool.

In [ ]:
import requests
import json
from packaging.version import Version, InvalidVersion


In [ ]:
# Development envirionment
API_URL_BASE = "https://dev.tools-registry.eosc-data-commons.eu/api/v1/"
# Demo environment
#API_URL_BASE = "https://tool-registry.eosc-data-commons.dansdemo.nl/api/v1/"
# Local env
#API_URL_BASE = "http://localhost:8080/api/v1/"
API_URL_TOOLS = f"{API_URL_BASE}tools/"
TOOL_REGISTRY_VERSION = "0.2.13"

## Getting access credentials
Before publishing to the tool registry an EGI access token is needed. This can be obtained from [https://aai-dev.egi.eu/token/](https://aai-dev.egi.eu/token/). 
Once obtained copy and paste in the cell below. The token is only valid for 1 hour.

In [ ]:
TOKEN = ""

Below example of Dimuon Spectrum Analysis notebook used by CernBox VRE. We try to capture the inputs with slots e.g. here we have one input slot which is a CSV file. Similarly, possible outputs are also described as slots. `raw_defiition` in this example is left empty since location points to the python notebook. 


In [ ]:
tool = {
    "uri": "https://raw.githubusercontent.com/dpiparo/swanExamples/refs/heads/master/notebooks/CMSDimuon_py.ipynb",
    "name": "Dimuon Spectrum Analysis",
    "version": "1",
    "location": "https://raw.githubusercontent.com/dpiparo/swanExamples/refs/heads/master/notebooks/CMSDimuon_py.ipynb",
    "description": "Loads CMS Run2010B dimuon collision event data from a CSV file into CERN ROOT TTrees, computes invariant mass spectra for muon pairs, filters opposite-charge muon events, generates histograms of the dimuon mass spectrum, and visualises resonances such as the J/Psi particle using ROOT canvases.",
    "keywords": ["cms", "dimuon"],
    "tags": ["cernbox", ""],
    "license": "LGPL-2.1",
    "types": ["python_notebook", "cernbox"],
    "input_slots":[{
            "id": 0,
            "name": "dimuon_collision_events",
            "optional": False,
            "description": "CSV file containing dimuon collision event data with particle momentum, energy, and charge information.",
            "default": None,
            "file_formats":["csv"]
        }],
    "output_file_formats":["root","png","pdf"],
    "output_slots":[{
            "id":0,
            "name": "root_binary",
            "description": "ROOT binary file containing TTrees and histograms generated during the analysis.",
            "file_formats":["root"]
            },
            {
            "id":1,
            "name": "visualisation_images",
            "description": "Rendered histogram and spectrum visualisation images.",
            "file_formats":["png"]
            },
            {
            "id":2,
            "name": "exported_plots",
            "description": "Exported publication-style plots and histogram visualisations.",
            "file_formats":["pdf"]
            }],
    "input_file_formats": ["csv"],
    "ouput_file_formats": ["root", "png", "pdf"],
    "raw_definition": {}
}   

## Check tool registry version
This notebook was created for tool-registry version > `0.2.12`. Different versions might not have the same schema!

In [ ]:
response = requests.get(f"{API_URL_BASE}health")
if response.status_code in (200, 201):
    data = response.json()
    version = data.get("version", None)    
    try:
        if version and Version(version) >= Version(TOOL_REGISTRY_VERSION):
            print(f"Correct tool registry version: {version}")
        else:
            print(f"Incorrect or outdated version for tool registry: {response.text}")

    except InvalidVersion:
        print(f"Invalid version format: {version}")
        
else:
    print(f"Failure: {response.status_code}, message: {response.text}")

## Publishing your tool
Next we can go ahead and submit the tool using a POST command.

In [ ]:
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
}
tool_id = None
response = requests.post(API_URL_TOOLS, json=tool, headers=headers)
if response.status_code in (200, 201):
    print("Success")
    tool_id = response.json()["tool_id"]
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    print(f"Created tool at: {tool_url}")
if response.status_code == 400:
    data = response.json()
    if data["detail"] and data["detail"]["existing_tool_id"]:
        tool_id = data["detail"]["existing_tool_id"]
        tool_url = f"{API_URL_TOOLS}{tool_id}"
        print(f"Tool already exists with ID: {tool_url}")

if not tool_id:
    print(f"Failure: {response.status_code}")
    print(response.text)

## Check your tool in the registry

On `Success`, the registry will respond with the tool ID. This is used to check/update/delete the tool in the future. 
Checking that your tool exists in the registry is a matter of calling the correct URL.

In [ ]:
if tool_id:
    
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.get(tool_url)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")

## Update your tool information
It is also possible to update the tool inforamtion after being created. Updating tools is only allowed by the user that created the tool. This is done similary to creating a tool but instead we only provide the fields needing updating. 
In this example we will update the `name` and `license`

In [ ]:
if tool_id:
    tool_update = {
        
    }
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.patch(tool_url, json=tool_update, headers=headers)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")

## Delete your tool from registry

It is also possible to delete the tool from the registry. This is only allowed by the user that created the tool.

> ⚠️ **Warning:** Proceed with caution. This action will delete your tool from the registry.

In [ ]:
if tool_id:
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.delete(tool_url, headers=headers)
    if tool_response.status_code in (200, 201):
        print(json.dumps(tool_response.json(), indent=4))
    else:
        print(f"Failure: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")

## Verify tool has been deleted
Use the same query as before to check if the tool exists (it should not!)

In [ ]:
if tool_id:
    
    tool_url = f"{API_URL_TOOLS}{tool_id}"
    tool_response = requests.get(tool_url)
    if tool_response.status_code == 404:
        print(f"Tool {tool_id} not in registry.")
    else:
        print(f"Status: {tool_response.status_code}")
        print(tool_response.text)
else:
    print("Tool was not created in previous step!")